In [ ]:
# Cell 1 - Setup
import chi, os, time, datetime
from chi import lease, server, network, context

context.version = "1.0"
context.choose_project()   # select CHI-251409 from dropdown
context.choose_site(default="KVM@TACC")
print("Setup done")

# Cell 2 - Names 
suffix      = "proj19"
lease_name  = f"bestshot-k8s-{suffix}"
node_name   = f"node-bestshot-{suffix}"
volume_name = f"bestshot-vol-{suffix}"

print(f"Lease name:  {lease_name}")
print(f"Node name:   {node_name}")
print(f"Volume name: {volume_name}")

# Cell 3 - Create lease
l = lease.Lease(
    lease_name,
    duration=datetime.timedelta(days=5)
)

l.add_flavor_reservation(
    id=chi.server.get_flavor_id("m1.medium"),
    amount=1
)

l.submit(idempotent=True)
print("Lease submitted! Waiting...")
l.show()

# Cell 4 - Launch VMs
flavor = l.get_reserved_flavors()[0].name
print(f"Using flavor: {flavor}")

node = server.Server(
    node_name,
    image_name="CC-Ubuntu24.04",
    flavor_name=flavor
)
node.submit(idempotent=True)
print("Waiting for node to boot...")
node.wait()
print("✅ Node is ready!")

# Cell 5 - Assign Floating IP
node.associate_floating_ip()
node.refresh()
fip = node.get_floating_ip()
print(f"✅ Floating IP: {fip}")
print(f"SSH command: ssh -i ~/.ssh/id_rsa_chameleon cc@{fip}")

# Cell 6 - Security groups
security_groups = [
    ("allow-ssh",   22,    "SSH access"),
    ("allow-http",  80,    "HTTP access"),
    ("allow-30283", 30283, "Immich NodePort"),
    ("allow-30500", 30500, "MLflow NodePort"),
    ("allow-6443",  6443,  "K8S API server"),
]

for sg_name, port, desc in security_groups:
    sg = network.SecurityGroup({"name": sg_name, "description": desc})
    sg.add_rule("ingress", "tcp", port)
    sg.submit(idempotent=True)
    node.add_security_group(sg_name)
    print(f"✅ Added: {sg_name} (port {port})")

# Cell 7 - Block volume
conn = chi.connection()

vol = conn.block_storage.create_volume(
    name=volume_name,
    size=20
)
print(f"Volume created: {volume_name}")
print("Waiting for volume to be ready...")

for i in range(24):
    vol = conn.block_storage.get_volume(vol.id)
    print(f"Volume status: {vol.status}")
    if vol.status == "available":
        print("✅ Volume ready!")
        break
    time.sleep(10)

conn.compute.create_volume_attachment(
    node.id,
    volumeId=vol.id
)
print(f"✅ Volume attached to node!")

# Cell 8 - Summary
node.refresh()
fip = node.get_floating_ip()

print(f"\n{'='*55}")
print(f"  BESTSHOT INFRASTRUCTURE READY")
print(f"{'='*55}")
print(f"  Node:        {node_name}")
print(f"  Floating IP: {fip}")
print(f"  Volume:      {volume_name} (20GB)")
print(f"{'='*55}")
print(f"\n  SSH into your node:")
print(f"  ssh -i ~/.ssh/id_rsa_chameleon cc@{fip}")
print(f"\n  After SSH, run:")
print(f"  1. Format volume (see README)")
print(f"  2. bash infra/k8s/setup_k8s.sh")
print(f"  3. kubectl apply -f infra/k8s/platform/")
print(f"  4. kubectl apply -f infra/k8s/app/")
print(f"\n  Immich:  http://{fip}:30283")
print(f"  MLflow:  http://{fip}:30500")
print(f"{'='*55}")